# Milestone 1 — Student A: HPatches data and SIFT/ORB baselines
**CSE 681, Project 26 — Owner: Student A (Data & Baseline Lead)**

All helper functions are defined inside this notebook. Required packages: NumPy, pandas, OpenCV, Matplotlib and IPython; Python standard-library modules handle files, ZIP extraction and timing. No neural-network framework is needed.

Run top to bottom in Colab or Jupyter. Outputs start empty: no performance numbers have been invented. The experiment uses real HPatches images only.

**Main settings retained from the uploaded notebook:** 108 sequences after eight named exclusions, native image resolution, up to 2,000 retained keypoints, mutual nearest-neighbor matching and RANSAC. These are project settings, not a claim of exact paper reproduction. They differ from the earlier 20-pair, 1,000-keypoint pilot; do not pool their results.

The full run evaluates 540 pairs per method when the complete dataset is present. `MAX_SEQUENCES_PER_KIND` can select a deterministic real-data subset; all captions/counts are generated from the actual selection. Runtime depends on the device and image sizes.

**Outputs:** dataset and baseline LaTeX tables; per-pair and per-run CSVs; MMA plots and match examples; settings, environment and sequence manifest. Student C may review these later; this notebook contains Student A work only.

In [ ]:
from pathlib import Path
import sys, os, json, time, zipfile, platform, subprocess
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/DIP Project/Compact_Feature_Matching')
    DATA = Path('/content/hpatches_student_a')
else:
    PROJECT = Path.cwd() / 'Compact_Feature_Matching'  # Edit for your local machine.
    DATA = PROJECT / 'data' / 'extracted'

# Edit only this path if the archive is stored elsewhere.
ZIP_PATH = PROJECT / 'data' / 'hpatches-sequences-release.zip'
OUT_ROOT = PROJECT / 'milestone1' / 'student_a_standalone'
DIRS = {name: OUT_ROOT/name for name in ['tables','figures','results']}
for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

SEED = 42
N_FEATURES = 2000
MAX_SEQUENCES_PER_KIND = None  # None: all eligible scenes; 2: two real scenes/category.
TIMING_REPEATS = 3
WARMUP_RUNS = 1  # Per method on each new image pair.
RANSAC_THRESHOLD = 3.0
MMA_THRESHOLDS = range(1,11)
HOMOG_THRESHOLDS = [1,3,5]
cv2.setNumThreads(1)
cv2.setRNGSeed(SEED)
np.random.seed(SEED)
print('Project:', PROJECT)
print('Dataset ZIP:', ZIP_PATH)
print('Output folder:', OUT_ROOT)
print('OpenCV:', cv2.__version__)

## 1. Read real HPatches sequences
The ZIP stays on Drive; extracting to Colab's temporary disk makes repeated reads faster. A marker avoids repeating extraction in one session. On a new runtime, extract again. No automatic network download is needed.
The eight exclusions below follow the commonly used reduced HPatches sequence set. Missing or corrupt files stop the run instead of silently changing its denominator.

In [ ]:
DATA.mkdir(parents=True, exist_ok=True)
marker = DATA/'.extraction_complete'
if not marker.exists():
    if not ZIP_PATH.is_file():
        raise FileNotFoundError(f'Upload the full-sequences ZIP to: {ZIP_PATH}')
    root = DATA.resolve()
    with zipfile.ZipFile(ZIP_PATH) as archive:
        for member in archive.namelist():
            if not (root/member).resolve().is_relative_to(root):
                raise ValueError('Archive contains an invalid extraction path.')
        archive.extractall(DATA)
    marker.touch()

EXCLUDED = {'i_contruction','i_crownnight','i_dc','i_pencils','i_whitebuilding',
            'v_artisans','v_astronautis','v_talent'}
ALL_SEQS = sorted({p.parent for p in DATA.rglob('1.ppm')
                   if p.parent.name.startswith(('i_','v_'))})
def seq_kind(folder):
    return 'illumination' if Path(folder).name.startswith('i_') else 'viewpoint'
SEQS = [s for s in ALL_SEQS if s.name not in EXCLUDED]
if MAX_SEQUENCES_PER_KIND is not None:
    if not isinstance(MAX_SEQUENCES_PER_KIND,int) or MAX_SEQUENCES_PER_KIND < 1:
        raise ValueError('MAX_SEQUENCES_PER_KIND must be None or a positive integer.')
    SEQS = [s for kind in ['illumination','viewpoint']
            for s in [x for x in SEQS if seq_kind(x)==kind][:MAX_SEQUENCES_PER_KIND]]
if not SEQS: raise FileNotFoundError('No full-image sequences were found.')
for folder in SEQS:
    required=[folder/f'{i}.ppm' for i in range(1,7)]+[folder/f'H_1_{i}' for i in range(2,7)]
    for path in required:
        if not path.is_file(): raise FileNotFoundError(path)
manifest=pd.DataFrame([{'scene':s.name,'category':seq_kind(s)} for s in SEQS])
manifest.to_csv(DIRS['results']/'m1_sequences.csv',index=False)
print('Available scenes:',len(ALL_SEQS))
print('Selected scenes:',len(SEQS),'| pairs per method:',5*len(SEQS))
display(manifest.groupby('category').size().rename('scenes'))

## 2. Image loading, tables and dataset examples
A sequence contains six views of one scene. `H_1_i` maps reference-image coordinates into image i. Images remain at native resolution in this notebook, so no homography resize correction is necessary.
The repository MIT license covers its software/documentation; retain original image-source attributions and check their usage terms. Do not label all images unrestricted simply because the repository code is MIT.

In [ ]:
def load_sequence(folder):
    folder=Path(folder)
    images=[]
    for i in range(1,7):
        image=cv2.imread(str(folder/f'{i}.ppm'),cv2.IMREAD_GRAYSCALE)
        if image is None: raise ValueError(f'Cannot read {folder}/{i}.ppm')
        images.append(image)
    transforms={1:np.eye(3)}
    for i in range(2,7):
        H=np.loadtxt(folder/f'H_1_{i}')
        if H.shape!=(3,3) or not np.isfinite(H).all():raise ValueError(f'Invalid H in {folder}')
        transforms[i]=H
    return images,transforms

def save_latex(frame,path,caption,label,index=False):
    # Plain Python exporter: no template engine or extra dependency required.
    data=frame.reset_index() if index else frame.copy()
    escapes={'\\':r'\textbackslash{}','&':r'\&','%':r'\%','$':r'\$',
             '#':r'\#','_':r'\_','{':r'\{','}':r'\}','~':r'\textasciitilde{}','^':r'\textasciicircum{}'}
    def escape(value):
        if isinstance(value,(float,np.floating)):
            value='--' if pd.isna(value) else f'{value:.3f}'
        return ''.join(escapes.get(ch,ch) for ch in str(value))
    lines=[r'\begin{table}[htbp]',r'\centering',r'\small',
           r'\caption{'+escape(caption)+'}',r'\label{'+label+'}',
           r'\begin{tabular}{@{}'+'l'*len(data.columns)+r'@{}}',r'\toprule',
           ' & '.join(escape(c) for c in data.columns)+r' \\',r'\midrule']
    for row in data.itertuples(index=False,name=None):
        lines.append(' & '.join(escape(v) for v in row)+r' \\')
    lines += [r'\bottomrule',r'\end{tabular}',r'\end{table}']
    Path(path).write_text('\n'.join(lines)+'\n',encoding='utf-8')

shapes=[]
for folder in SEQS:
    image=cv2.imread(str(folder/'1.ppm'),0)
    if image is None:raise ValueError(f'Unreadable reference image: {folder}')
    shapes.append(image.shape)
stats=pd.DataFrame([
    {'Item':'Available / selected sequences','Value':f'{len(ALL_SEQS)} / {len(SEQS)}'},
    {'Item':'Selected illumination / viewpoint','Value':f'{sum(seq_kind(s)=="illumination" for s in SEQS)} / {sum(seq_kind(s)=="viewpoint" for s in SEQS)}'},
    {'Item':'Pairs per method','Value':str(5*len(SEQS))},
    {'Item':'Median reference H x W','Value':f'{int(np.median([s[0] for s in shapes]))} x {int(np.median([s[1] for s in shapes]))}'},
    {'Item':'Repository license','Value':'MIT software/documentation; retain image-source terms'},
])
display(stats)
save_latex(stats,DIRS['tables']/'m1_dataset.tex','HPatches selection used in this run.','tab:data_stats')

In [ ]:
examples=[next((s for s in SEQS if seq_kind(s)==kind),None)
          for kind in ['illumination','viewpoint']]
examples=[s for s in examples if s is not None]
fig,axes=plt.subplots(len(examples),7,figsize=(20,3*len(examples)),squeeze=False)
for r,folder in enumerate(examples):
    images,Hs=load_sequence(folder)
    for i,image in enumerate(images):
        axes[r,i].imshow(image,cmap='gray');axes[r,i].set_title(f'{folder.name} #{i+1}',fontsize=8)
    h,w=images[5].shape
    warped=cv2.warpPerspective(images[0],Hs[6],(w,h))
    axes[r,6].imshow(np.dstack([warped,images[5],warped]))
    axes[r,6].set_title('Reference warp vs image 6',fontsize=8)
for ax in axes.ravel():ax.axis('off')
plt.tight_layout();plt.savefig(DIRS['figures']/'m1_hpatches_examples.png',dpi=160);plt.show()

## 3. Matching and geometry helpers (defined here)
SIFT descriptors use Euclidean/L2 distance. Default ORB descriptors use Hamming distance. `crossCheck=True` keeps mutual nearest-neighbor matches. Features are explicitly capped at N_FEATURES after detection, with descriptors kept in the same order.

Forward transfer error uses H_gt to score a match, without using H_gt to generate it. MMA@t is the pair-level fraction with error **≤ t pixels**, then averaged equally over pairs. Zero-match pairs score 0. These inclusive thresholds are declared project conventions; they differ from the earlier strict <3-pixel pilot.
RANSAC uses H_est to identify inliers. Corner error averages distances between four corners projected by H_est and H_gt. Missing/invalid transforms count as failed registrations; post-inlier precision is undefined when no inliers remain.

In [ ]:
def extract_features(image,method):
    extractor=cv2.SIFT_create(nfeatures=N_FEATURES) if method=='SIFT' else cv2.ORB_create(nfeatures=N_FEATURES)
    keypoints,descriptors=extractor.detectAndCompute(image,None)
    if descriptors is None:return [],None
    indices=sorted(range(len(keypoints)),key=lambda i:keypoints[i].response,reverse=True)[:N_FEATURES]
    return [keypoints[i] for i in indices],np.ascontiguousarray(descriptors[indices])

def project_points(points,H):
    points=np.asarray(points,float).reshape(-1,2)
    q=np.column_stack([points,np.ones(len(points))])@H.T
    valid=np.isfinite(q).all(axis=1)&(np.abs(q[:,2])>1e-10)
    projected=np.full((len(points),2),np.nan)
    projected[valid]=q[valid,:2]/q[valid,2:3]
    return projected

def corner_error(H_est,H_gt,width,height):
    if H_est is None or not np.isfinite(H_est).all() or np.linalg.matrix_rank(H_est)<3:return np.inf
    corners=np.array([[0,0],[width-1,0],[width-1,height-1],[0,height-1]],float)
    a,b=project_points(corners,H_est),project_points(corners,H_gt)
    if not (np.isfinite(a).all() and np.isfinite(b).all()):return np.inf
    return float(np.linalg.norm(a-b,axis=1).mean())

def register_pair(image1,image2,method):
    cv2.setRNGSeed(SEED)
    start=time.perf_counter()
    k1,d1=extract_features(image1,method);k2,d2=extract_features(image2,method)
    extracted=time.perf_counter()
    norm=cv2.NORM_L2 if method=='SIFT' else cv2.NORM_HAMMING
    matches=[] if d1 is None or d2 is None else list(cv2.BFMatcher(norm,crossCheck=True).match(d1,d2))
    matched=time.perf_counter()
    p1=np.array([k1[m.queryIdx].pt for m in matches],float).reshape(-1,2)
    p2=np.array([k2[m.trainIdx].pt for m in matches],float).reshape(-1,2)
    H,mask=None,None
    if len(matches)>=4:
        H,mask=cv2.findHomography(p1,p2,cv2.RANSAC,RANSAC_THRESHOLD,maxIters=2000,confidence=0.995)
    ended=time.perf_counter()
    inliers=np.zeros(len(matches),bool) if H is None or mask is None else mask.ravel().astype(bool)
    sizes=[(d.nbytes/len(d)) for d in [d1,d2] if d is not None and len(d)]
    return dict(k1=k1,k2=k2,matches=matches,p1=p1,p2=p2,H=H,inliers=inliers,
        bytes_per_desc=float(np.mean(sizes)) if sizes else np.nan,
        extract_ms=1000*(extracted-start)/2,  # mean per image, two images extracted
        match_ms=1000*(matched-extracted),geometry_ms=1000*(ended-matched),total_ms=1000*(ended-start))

def score_pair(result,H_gt,shape):
    error=np.linalg.norm(project_points(result['p1'],H_gt)-result['p2'],axis=1)
    error[~np.isfinite(error)]=np.inf
    count=len(error);inliers=result['inliers'];ni=int(inliers.sum())
    ce=corner_error(result['H'],H_gt,shape[1],shape[0])
    scores={f'mma@{t}':float(np.mean(error<=t)) if count else 0.0 for t in MMA_THRESHOLDS}
    scores.update({f'hacc@{t}':float(ce<=t) for t in HOMOG_THRESHOLDS})
    scores.update(n_matches=count,inliers=ni,correct_matches_3px=int((error<=3).sum()),
        post_precision_3px=float((error[inliers]<=3).mean()) if ni else np.nan,
        corner_error_px=ce,valid_homography=bool(np.isfinite(ce)))
    return scores

## 4. Run SIFT/ORB on the real selected scenes
For every sequence, pair image 1 with 2–6. One warm-up and TIMING_REPEATS measured runs are used for each pair/method. Record median times, with extraction shown as milliseconds **per image**; matching/geometry/total are per pair. Disk reads, plotting and ground-truth scoring are excluded. Extractor construction is included within extraction in this implementation. Absolute speed depends on hardware.
Save progress after each scene. Rerunning this cell restarts the experiment and overwrites its output files. Failures remain in results. This is evaluation-only: no network training, no model checkpoint and no train/test split is created here. Future learned-method training must use data disjoint from its final evaluation scenes.

In [ ]:
rows=[];time_rows=[]
for scene_index,folder in enumerate(SEQS,start=1):
    images,Hs=load_sequence(folder)
    for target in range(2,7):
        for method in ['SIFT','ORB']:
            for _ in range(WARMUP_RUNS):register_pair(images[0],images[target-1],method)
            trials=[register_pair(images[0],images[target-1],method) for _ in range(TIMING_REPEATS)]
            result=trials[0]
            timing={name:float(np.median([r[name] for r in trials]))
                    for name in ['extract_ms','match_ms','geometry_ms','total_ms']}
            rows.append(dict(method=method,scene=folder.name,kind=seq_kind(folder),target=target,
                keypoints1=len(result['k1']),keypoints2=len(result['k2']),
                bytes_per_desc=result['bytes_per_desc'],**score_pair(result,Hs[target],images[0].shape),**timing))
            for run_number,r in enumerate(trials,start=1):
                time_rows.append(dict(method=method,scene=folder.name,target=target,run=run_number,
                    **{name:r[name] for name in timing}))
    pd.DataFrame(rows).to_csv(DIRS['results']/'m1_baseline_pairs.csv',index=False)
    pd.DataFrame(time_rows).to_csv(DIRS['results']/'m1_timing_runs.csv',index=False)
    print(f'{scene_index}/{len(SEQS)} completed: {folder.name}')
df=pd.DataFrame(rows)
assert len(df)==len(SEQS)*5*2
print('Result rows:',len(df))
display(df.head())

## 5. Baseline table and mean matching-accuracy plot
Each pair has equal weight within overall/illumination/viewpoint groups. Scores are fractions (0.8 means 80%). H-acc includes failed transformations in the denominator. Median corner error is over valid estimates only; valid-estimate rate is reported alongside it. Post-RANSAC precision excludes undefined entries, and its contributing-pair count is disclosed.
The LaTeX file can be included with `\input{tables/m1_baseline.tex}` after adding booktabs to the report. No unverified published scores or exact-reproduction claims are inserted.

In [ ]:
summary_rows=[]
for method in ['SIFT','ORB']:
    method_df=df[df.method==method]
    for kind in ['overall','illumination','viewpoint']:
        g=method_df if kind=='overall' else method_df[method_df.kind==kind]
        if g.empty:continue
        row=dict(method=method,kind=kind,pairs=len(g))
        keys=[f'mma@{t}' for t in MMA_THRESHOLDS]+[f'hacc@{t}' for t in HOMOG_THRESHOLDS]
        row.update({key:float(g[key].mean()) for key in keys})
        row.update(n_matches=g.n_matches.mean(),extract_ms=g.extract_ms.mean(),
            total_ms=g.total_ms.median(),bytes_per_desc=g.bytes_per_desc.mean(),
            valid_homography_rate=g.valid_homography.mean(),
            median_corner_valid_px=g.loc[np.isfinite(g.corner_error_px),'corner_error_px'].median(),
            post_precision_3px=g.post_precision_3px.mean(),post_precision_pairs=g.post_precision_3px.notna().sum())
        summary_rows.append(row)
summ=pd.DataFrame(summary_rows).set_index(['method','kind'])
summ.to_csv(DIRS['results']/'m1_baseline_summary.csv')
cols=['pairs','mma@1','mma@3','mma@5','mma@10','hacc@1','hacc@3','hacc@5',
      'n_matches','extract_ms','total_ms','bytes_per_desc']
table=summ[cols].round(3).reset_index()
display(table)
caption=f'Student A baseline: {len(SEQS)} sequences, {5*len(SEQS)} pairs per method, native resolution, {N_FEATURES} features requested and capped, mutual matching. Scores are fractions; extraction time is per image and total time is per pair.'
save_latex(table,DIRS['tables']/'m1_baseline.tex',caption,'tab:m1_baseline')

fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,kind in zip(axes,['overall','illumination','viewpoint']):
    for method in ['SIFT','ORB']:
        if (method,kind) in summ.index:
            ax.plot(list(MMA_THRESHOLDS),[summ.loc[(method,kind),f'mma@{t}'] for t in MMA_THRESHOLDS],marker='o',label=method)
    ax.set_title(kind);ax.set_xlabel('Forward error threshold (pixels)');ax.set_ylabel('Mean matching accuracy')
    ax.set_ylim(0,1);ax.grid(alpha=.3);ax.legend()
plt.tight_layout();plt.savefig(DIRS['figures']/'m1_baseline_mma.pdf');plt.show()

## 6. Qualitative viewpoint example
Choose the first selected viewpoint sequence and pair 1→5 deterministically. Display up to 40 RANSAC inliers. This is an illustrative viewpoint example, not a claim that it is the hardest pair or that every retained match is correct. Visualization reruns the pipeline and is excluded from benchmark timing.

In [ ]:
folder=next((s for s in SEQS if seq_kind(s)=='viewpoint'),None)
if folder is not None:
    images,Hs=load_sequence(folder)
    fig,axes=plt.subplots(2,1,figsize=(14,10))
    for ax,method in zip(axes,['SIFT','ORB']):
        r=register_pair(images[0],images[4],method)
        kept=[m for m,keep in zip(r['matches'],r['inliers']) if keep]
        shown=sorted(kept,key=lambda m:m.distance)[:40]
        picture=cv2.drawMatches(images[0],r['k1'],images[4],r['k2'],shown,None,matchColor=(0,255,0),flags=2)
        ce=corner_error(r['H'],Hs[5],images[0].shape[1],images[0].shape[0])
        ax.imshow(cv2.cvtColor(picture,cv2.COLOR_BGR2RGB))
        ax.set_title(f'{method}: {len(r["matches"])} matches, {len(kept)} inliers, corner error {ce:.3f}px | {folder.name} 1→5')
        ax.axis('off')
    plt.tight_layout();plt.savefig(DIRS['figures']/'m1_baseline_matches.png',dpi=150);plt.show()

## 7. Export report sentences and reproducibility records
The generated sentences describe only results computed above. Add exact CPU/RAM and software versions to your report. Colab's CPU can differ between sessions; compare methods within the same run.

In [ ]:
for method in ['SIFT','ORB']:
    o=summ.loc[(method,'overall')]
    print(f'{method}: {int(o["pairs"])} pairs; MMA@3={o["mma@3"]:.3f}; H-acc@3={o["hacc@3"]:.3f}; mean matches/pair={o["n_matches"]:.1f}; mean extraction={o["extract_ms"]:.3f} ms/image; median total={o["total_ms"]:.3f} ms/pair.')
settings=dict(seed=SEED,n_features=N_FEATURES,excluded=sorted(EXCLUDED),
    sequences=[s.name for s in SEQS],max_sequences_per_kind=MAX_SEQUENCES_PER_KIND,
    resolution='native',matching='mutual nearest neighbors',error='forward transfer',
    threshold_comparison='<=',ransac_threshold=RANSAC_THRESHOLD,ransac_max_iters=2000,
    ransac_confidence=0.995,warmups=WARMUP_RUNS,timing_repeats=TIMING_REPEATS,
    zero_match_mma=0,failed_homography_accuracy=0,empty_inlier_precision='undefined',
    opencv_threads=cv2.getNumThreads())
environment=dict(python=sys.version,platform=platform.platform(),processor=platform.processor(),
    opencv=cv2.__version__,numpy=np.__version__,pandas=pd.__version__)
(DIRS['results']/'m1_protocol.json').write_text(json.dumps(settings,indent=2))
(DIRS['results']/'m1_environment.json').write_text(json.dumps(environment,indent=2))
(DIRS['results']/'requirements_actual.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('Saved all outputs to:',OUT_ROOT)

## Student A deliverable checklist
- Dataset summary and selected-sequence manifest; real image examples and checked source terms.
- SIFT/ORB per-pair results, aggregate table and MMA plot, with failures included.
- Exact evaluation settings, runtime versions, hardware description and measured timing protocol.
- Dataset and classical-baseline paragraphs for report Section 3; do not present these full-set results as the earlier 20-pair pilot.
- Literature table: at least four references with at least two from 2023–2026; summarize their relevance in your own words.
- Team repository and README, plus an accurate AI-assistance statement. Student C can independently review these metrics. Student B handles learned-model design/training separately.

Sources: [HPatches](https://github.com/hpatches/hpatches-dataset), [D2-Net sequence benchmark](https://github.com/mihaidusmanu/d2-net/tree/master/hpatches_sequences), [OpenCV matcher](https://docs.opencv.org/4.x/dc/dc3/tutorial_py_matcher.html), [SuperPoint](https://arxiv.org/abs/1712.07629), [ALIKED](https://arxiv.org/abs/2304.03608), [XFeat](https://arxiv.org/abs/2404.19174).

This notebook does not compare against hard-coded paper scores: resize, detector selection, matching, corner definitions and software must all be verified before claiming exact reproduction.